In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl

REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "common" / "paths.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.paths import ROOT

from common.dataset import make_loader
from common.metrics import dump_json, load_weight_only, regression_frames, save_weight_only, summarize_regression
from common.models import EmeanRegressor
from common.train import BestStateCallback, EpochRecorder

DATASET_NAME = "kidney"
EXPERIMENT_NAME = "real_swe"
INPUT_MODE = "real_swe"
NOTEBOOK_DIR = ROOT / "experiment" / "kidney" / "03_comparison"
PAIR_CSV = ROOT / "experiment" / "kidney" / "01_data_processing" / "cache" / "pairs_seed_42.csv"
EXTERNAL_CSV = ROOT / "experiment" / "kidney" / "01_data_processing" / "cache" / "external_pairs.csv"
training = False
SEED = 42
BATCH_SIZE = 16
MAX_EPOCHS = 60
NUM_WORKERS = 4
LEARNING_RATE = 1e-3
WEIGHT_PATH = NOTEBOOK_DIR / "real_swe_emean_head.pt"
METRICS_PATH = NOTEBOOK_DIR / "metrics_real_swe.json"

pl.seed_everything(SEED, workers=True)
if not PAIR_CSV.is_file():
    raise FileNotFoundError(PAIR_CSV)
frame = pd.read_csv(PAIR_CSV)
train_frame = frame.loc[frame["split"] == "train"].copy()
validation_frame = frame.loc[frame["split"] == "val"].copy()
test_frame = frame.loc[frame["split"] == "test"].copy()
external_frame = pd.read_csv(EXTERNAL_CSV)
train_loader = make_loader(train_frame, BATCH_SIZE, True, True, NUM_WORKERS)
train_evaluation_loader = make_loader(train_frame, BATCH_SIZE, False, False, NUM_WORKERS)
validation_loader = make_loader(validation_frame, BATCH_SIZE, False, False, NUM_WORKERS)
test_loader = make_loader(test_frame, BATCH_SIZE, False, False, NUM_WORKERS)
external_loader = make_loader(external_frame, BATCH_SIZE, False, False, NUM_WORKERS)
log_targets = np.log1p(train_frame["emean"].to_numpy(dtype=float))
target_mean = float(log_targets.mean())
target_std = float(log_targets.std(ddof=1))
if not np.isfinite(target_std) or target_std <= 0:
    raise ValueError("Invalid Emean standard deviation")

In [ ]:
class EmeanRegressionLightning(pl.LightningModule):
    def __init__(
        self,
        input_mode,
        target_mean,
        target_std,
        translation=None,
        learning_rate=1e-3,
    ):
        super().__init__()
        self.input_mode = input_mode
        self.translation = translation
        if self.translation is not None:
            self.translation.eval()
            for parameter in self.translation.parameters():
                parameter.requires_grad_(False)
        self.regressor = EmeanRegressor()
        self.register_buffer("target_mean", torch.tensor(float(target_mean)))
        self.register_buffer("target_std", torch.tensor(float(target_std)))
        self.learning_rate = learning_rate
        self.loss_function = nn.HuberLoss(delta=1.0)

    def on_train_epoch_start(self):
        if self.translation is not None:
            self.translation.eval()

    def _images(self, batch):
        if self.input_mode == "gray":
            return batch["gray"]
        if self.input_mode == "real_swe":
            return batch["swe"]
        if self.input_mode == "virtual_swe":
            with torch.no_grad():
                return self.translation(batch["gray"])
        raise ValueError(self.input_mode)

    def _standardized_prediction(self, batch):
        return self.regressor(self._images(batch))

    def forward(self, batch):
        standardized = self._standardized_prediction(batch)
        return torch.expm1(standardized * self.target_std + self.target_mean).clamp_min(0.0)

    def _loss(self, batch):
        target = (torch.log1p(batch["emean"]) - self.target_mean) / self.target_std
        return self.loss_function(self._standardized_prediction(batch), target)

    def training_step(self, batch, batch_index):
        loss = self._loss(batch)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_index):
        loss = self._loss(batch)
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_index):
        loss = self._loss(batch)
        prediction = self(batch)
        mae = torch.mean(torch.abs(prediction - batch["emean"]))
        self.log("test_loss", loss, on_epoch=True)
        self.log("test_mae", mae, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_index, dataloader_index=0):
        return {
            "prediction": self(batch).detach().cpu(),
            "target": batch["emean"].detach().cpu(),
            "image_name": list(batch["image_name"]),
            "patient_id": list(batch["patient_id"]),
        }

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.regressor.parameters(),
            lr=self.learning_rate,
            weight_decay=1e-4,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=6,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

In [ ]:
def collect_regression(outputs):
    predictions = torch.cat([output["prediction"] for output in outputs]).numpy()
    targets = torch.cat([output["target"] for output in outputs]).numpy()
    image_names = [name for output in outputs for name in output["image_name"]]
    patient_ids = [name for output in outputs for name in output["patient_id"]]
    return predictions, targets, image_names, patient_ids


def evaluate_regression(model, trainer, loader, output_prefix):
    outputs = trainer.predict(model, loader)
    predictions, targets, image_names, patient_ids = collect_regression(outputs)
    image_frame, patient_frame = regression_frames(
        predictions,
        targets,
        image_names,
        patient_ids,
    )
    image_frame.to_csv(NOTEBOOK_DIR / f"{output_prefix}_predictions.csv", index=False)
    patient_frame.to_csv(NOTEBOOK_DIR / f"{output_prefix}_patient_predictions.csv", index=False)
    return summarize_regression(image_frame, patient_frame)

In [ ]:
model = EmeanRegressionLightning(
    input_mode=INPUT_MODE,
    target_mean=target_mean,
    target_std=target_std,
    translation=None,
    learning_rate=LEARNING_RATE,
)
best = BestStateCallback("val_loss", "regressor")
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    deterministic=True,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
    enable_progress_bar=False,
    callbacks=[
        EpochRecorder(NOTEBOOK_DIR / f"{EXPERIMENT_NAME}_epoch_metrics.csv"),
        best,
        pl.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=12,
        ),
    ],
    num_sanity_val_steps=0,
)
if training:
    trainer.fit(model, train_loader, validation_loader)
    save_weight_only(model.regressor, WEIGHT_PATH)
    reloaded_regressor = EmeanRegressor()
    load_weight_only(reloaded_regressor, WEIGHT_PATH)
    model.regressor.load_state_dict(reloaded_regressor.state_dict())
else:
    if not WEIGHT_PATH.is_file():
        raise FileNotFoundError(WEIGHT_PATH)
    load_weight_only(model.regressor, WEIGHT_PATH)
trainer.validate(model, validation_loader)
trainer.test(model, test_loader)
metrics = {
    "train": evaluate_regression(model, trainer, train_evaluation_loader, f"{EXPERIMENT_NAME}_train"),
    "validation": evaluate_regression(model, trainer, validation_loader, f"{EXPERIMENT_NAME}_validation"),
    "internal_test": evaluate_regression(model, trainer, test_loader, f"{EXPERIMENT_NAME}_internal_test"),
}
trainer.test(model, external_loader)
metrics["external_test"] = evaluate_regression(
    model,
    trainer,
    external_loader,
    f"{EXPERIMENT_NAME}_external_test",
)
dump_json({"dataset": DATASET_NAME, "experiment": EXPERIMENT_NAME, "emean": metrics}, METRICS_PATH)